In [57]:
import pandas as pd
import numpy as np

response = pd.read_csv('csv_files/response.csv')
response.head()

,timestamp,year,program,beh_daily_use,beh_time_spent,beh_attendance,hours_per_week_raw,pu_useful,pu_efficiency,il_enjoyment,...,biu_intention,biu_voluntary,srl_strategy,srl_organize,srl_review,srl_persist,srl_evaluate,open_frustration,open_feature_request,open_srs_usage
0,9/6/2026 14:45,3rd year,Physics/Maths,3,3,2,12-Oct,3,3,3,...,4,3,4,4,4,3,4,Subscriptions,A feature to show what one should already know...,Decent
1,9/6/2026 15:00,6,BIT,4,4,2,20hrs,5,4,4,...,4,4,4,5,4,4,4,to pay,QNA,No
2,9/6/2026 15:05,6th sem,BIT,3,4,4,5,4,4,4,...,4,3,3,4,4,4,4,Giving short answers,Organizing my research,It helps in revising my study
3,9/6/2026 15:21,3rd year/6th sem,BIT,3,3,3,48 hrs,2,4,3,...,3,4,3,5,4,4,3,There unnecessary extra answers,Easier prompt to get enough required answer,No
4,9/6/2026 16:22,6th,BIT,3,3,1,12-Oct,2,3,2,...,2,2,2,3,2,2,2,Distraction,Idk,No


### Cleaning data

#### Year column

In [58]:
response['year'].value_counts()

year
6th                 4
6                   2
6th sem             2
3rd year            1
6th sem             1
3rd year/6th sem    1
3rd                 1
Six                 1
6th semester        1
6 semester          1
6 sem               1
Sixth               1
Name: count, dtype: int64

In [59]:
response['year'] = response['year'].str.strip()

year_map = {
    '6th' : '3rd',
    '6' : '3rd',
    '6th sem' : '3rd',
    '3rd year' : '3rd',
    '6th sem' : '3rd',
    '3rd year/6th sem' : '3rd',
    'Six' : '3rd',
    '6th semester' : '3rd',
    '6 semester' : '3rd',
    'Sixth' : '3rd',
    
}

response['year'] = response['year'].map(year_map).fillna(response['year'])
response['year'].value_counts()

year
3rd      16
6 sem     1
Name: count, dtype: int64

#### program column

In [60]:
response['program'].value_counts()

program
BIT              11
Physics/Maths     1
BIT               1
Bit               1
Bsc physics       1
BSc.CSIT          1
Name: count, dtype: int64

In [61]:
response['program'] = response['program'].str.strip()

program_map = {
    'BIT': 'BIT',
    'Bit': 'BIT',
    'bit': 'BIT',
    'Physics/Maths': 'BSc Physics',
    'Bsc physics': 'BSc Physics',
    'BSc.CSIT': 'BSc CSIT',
}

response['program'] = response['program'].map(program_map).fillna(response['program'])
response['program'].value_counts()

program
BIT            13
BSc Physics     2
BSc CSIT        1
Name: count, dtype: int64

In [62]:
response['program'].isna().sum()

np.int64(1)

In [63]:
response['program'] = response['program'].fillna('BSc CSIT')
response['program'].value_counts()

program
BIT            13
BSc Physics     2
BSc CSIT        2
Name: count, dtype: int64

#### Dropping unwanted responses

In [64]:
response.iloc[11]

timestamp                    9/7/2026 18:42
year                                  6 sem
program                                 BIT
beh_daily_use                             3
beh_time_spent                            3
beh_attendance                            3
hours_per_week_raw              2-3 per day
pu_useful                                 3
pu_efficiency                             3
il_enjoyment                              3
il_curiosity                              3
em_connection                             3
biu_intention                             3
biu_voluntary                             3
srl_strategy                              3
srl_organize                              3
srl_review                                3
srl_persist                               3
srl_evaluate                              3
open_frustration        Sometimes annoying 
open_feature_request                    Idk
open_srs_usage                          Idk
Name: 11, dtype: object

In [65]:
response.drop(11, inplace=True)
response.reset_index(drop=True, inplace=True) 

In [66]:
response.iloc[11]

timestamp                                         9/7/2026 22:22
year                                                         3rd
program                                                      BIT
beh_daily_use                                                  2
beh_time_spent                                                 2
beh_attendance                                                 2
hours_per_week_raw                                         3-Feb
pu_useful                                                      3
pu_efficiency                                                  2
il_enjoyment                                                   3
il_curiosity                                                   2
em_connection                                                  2
biu_intention                                                  2
biu_voluntary                                                  2
srl_strategy                                                   2
srl_organize             

### Looking at and cleaning hours students spent using digital tools

In [67]:
response['hours_per_week_raw'].unique()

<StringArray>
[                          '12-Oct',                            '20hrs',
                                '5',                           '48 hrs',
                              '10+',                           '3 hrs ',
                        '42 hours ',                             '8 10',
 'Not but probably more than 10hrs',                         '20 hours',
                            '3-Feb',                                '7',
                     '4-5 per day ',                          '5 hours',
                               '16']
Length: 15, dtype: str

In [68]:
hour_map = {
    '12-Oct': 11,        
    '3-Feb': 2.5,       
    '5': 5,
    '20hrs': 20,
    '10+': 10,
    '48 hrs': 48,
    '42 hours ': 42,
    '3 hrs ': 3,
    'Not but probably more than 10hrs': 10,
    '8 10': 9,          
    '20 hours': 20,
    '4-5 per day': 4.5,
    '7': 7,
    '16': 16,
    '5 hours': 5,
}

response['hours_per_week_num'] = response['hours_per_week_raw'].map(hour_map)

bins = [0, 5, 10, 20, np.inf]
labels = ['0-5', '6-10', '11-20', '20+']
response['hours_bucket'] = pd.cut(response['hours_per_week_num'], bins=bins, labels=labels, right=False)

response[['hours_per_week_raw', 'hours_per_week_num', 'hours_bucket']].head()

,hours_per_week_raw,hours_per_week_num,hours_bucket
0,12-Oct,11.0,11-20
1,20hrs,20.0,20+
2,5,5.0,6-10
3,48 hrs,48.0,20+
4,12-Oct,11.0,11-20


### Dropping unwanted columns

In [69]:
response.drop(columns=['timestamp']).head()

,year,program,beh_daily_use,beh_time_spent,beh_attendance,hours_per_week_raw,pu_useful,pu_efficiency,il_enjoyment,il_curiosity,...,srl_strategy,srl_organize,srl_review,srl_persist,srl_evaluate,open_frustration,open_feature_request,open_srs_usage,hours_per_week_num,hours_bucket
0,3rd,BSc Physics,3,3,2,12-Oct,3,3,3,4,...,4,4,4,3,4,Subscriptions,A feature to show what one should already know...,Decent,11.0,11-20
1,3rd,BIT,4,4,2,20hrs,5,4,4,3,...,4,5,4,4,4,to pay,QNA,No,20.0,20+
2,3rd,BIT,3,4,4,5,4,4,4,4,...,3,4,4,4,4,Giving short answers,Organizing my research,It helps in revising my study,5.0,6-10
3,3rd,BIT,3,3,3,48 hrs,2,4,3,3,...,3,5,4,4,3,There unnecessary extra answers,Easier prompt to get enough required answer,No,48.0,20+
4,3rd,BIT,3,3,1,12-Oct,2,3,2,4,...,2,3,2,2,2,Distraction,Idk,No,11.0,11-20


### Saving the clean responses to new csv

In [70]:
response.to_csv('csv_files/response_clean.csv', index = False)
print('Data cleaning complete')

Data cleaning complete
